In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import asyncio
import random
from agents import Agent, function_tool
from agents.extensions.handoff_prompt import prompt_with_handoff_instructions

@function_tool
def get_weather(city:str)->str:
    """Get the weather for a given city."""
    print(f"[debug] get_weather called with city: {city}")
    choices=["sunny","cloudy","rainy","snowy"]
    return f"The Weather in {city} is {random.choice(choices)}."

spanish_agent=Agent(
    name="spanish agent",
    handoff_description="A spanish speaking agent.",
    instructions=prompt_with_handoff_instructions(
        "You're speaking to a human, so be polite and "
    ),
    model="gpt-5-nano"
)

agent=Agent(
    name="Assistant",
    instructions=prompt_with_handoff_instructions("You're speaking to a human, so be polite and concise. if the user speak in spanish, handoff to the spanish agent."),
    model="gpt-5-nano",
    handoffs=[spanish_agent],
    tools=[get_weather]
)

In [3]:
# voice pipeline
from agents.voice import SingleAgentVoiceWorkflow, VoicePipeline
pipeline=VoicePipeline(workflow=SingleAgentVoiceWorkflow(agent))

In [4]:
import numpy as np
import sounddevice as sd
from agents.voice import AudioInput

buffer=np.zeros(24000*3,dtype=np.int16)
audio_input=AudioInput(buffer=buffer)
result=await pipeline.run(audio_input)

player=sd.OutputStream(samplerate=24000, channels=1, dtype=np.int16)
player.start()

async for event in result.stream():
    if event.type=="voice_stream_event_audio":
        player.write(event.data)